# VoiceStudio: Train XTTS-v2 on Google Colab

เทรนโมเดลเสียง XTTS-v2 ด้วย GPU ของ Google Colab

> **สำคัญ:** ก่อนเริ่ม ไปที่เมนู **Runtime > Change runtime type** แล้วเลือก **T4 GPU**

### ไฟล์ที่ต้องอัปโหลดก่อนเริ่ม:
1. `VoiceStudio.zip` — โค้ดโปรเจกต์ทั้งหมด
2. `japanese_voice_colab.zip` — ชุดข้อมูลเสียงที่เตรียมไว้แล้ว

ลากทั้ง 2 ไฟล์ไปวางในแถบไฟล์ด้านซ้ายของ Colab

In [ ]:
# 0. เช็ค GPU
!nvidia-smi

In [ ]:
# 1. เชื่อมต่อ Google Drive (เซฟโมเดลไว้ไม่ให้หาย)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. ติดตั้ง uv + แตกไฟล์โปรเจกต์ + ติดตั้ง dependencies
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os
os.environ['PATH'] = f"/root/.cargo/bin:{os.environ['PATH']}"

!unzip -q VoiceStudio.zip
%cd VoiceStudio
!uv sync

In [ ]:
# 3. แตกไฟล์ชุดข้อมูลเสียง
!unzip -q /content/japanese_voice_colab.zip -d data/ft/japanese_voice

In [ ]:
# 4. ดาวน์โหลดโมเดลต้นแบบ XTTS-v2 (~1.9 GB)
!uv run python -c "from TTS.api import TTS; TTS('tts_models/multilingual/multi-dataset/xtts_v2')"

In [ ]:
# 5. เทรนโมเดล!
# --language ja = ภาษาญี่ปุ่น (เปลี่ยนเป็น en ถ้าเสียงเป็นภาษาอังกฤษ)
# --epochs 30  = จำนวนรอบเทรน (เพิ่มได้ถ้าต้องการคุณภาพดีขึ้น)
!uv run python scripts/train_xtts.py \
    --dataset data/ft/japanese_voice \
    --language ja \
    --epochs 30 \
    --base-dir /root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2

In [ ]:
# 6. เซฟผลลัพธ์ลง Google Drive (เร็วกว่า files.download มาก!)
!cp -r training/run /content/drive/MyDrive/VoiceStudio_Checkpoints_v2
print('เซฟลง Google Drive เรียบร้อย! ไปดาวน์โหลดได้ที่ drive.google.com')